In [ ]:
!pip -q install rapidfuzz sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 11.4 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np

from rapidfuzz.distance import Levenshtein
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# =========================================================
# STEP 1: Rename folders
# =========================================================

base_dir = os.getcwd()

for folder in os.listdir(base_dir):
    old_path = os.path.join(base_dir, folder)

    if os.path.isdir(old_path) and folder.startswith("submission - "):
        new_name = folder.replace("submission - ", "")
        new_path = os.path.join(base_dir, new_name)

        if not os.path.exists(new_path):
            os.rename(old_path, new_path)
            print(f"Renamed: {folder} -> {new_name}")

print("\nFolder renaming complete.\n")

# =========================================================
# STEP 2: Load target file
# =========================================================

target_path = os.path.join(base_dir, "test_target.csv")

target_df = pd.read_csv(target_path)

# Lowercase column names
target_df.columns = target_df.columns.str.lower()

print("Target columns:")
print(target_df.columns.tolist())

# =========================================================
# STEP 3: Load embedding model
# =========================================================

model = SentenceTransformer("all-MiniLM-L6-v2")

# =========================================================
# STEP 4: Helper functions
# =========================================================

def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def compute_metrics(true_series, pred_series):

    true_texts = true_series.map(normalize_text).tolist()
    pred_texts = pred_series.map(normalize_text).tolist()

    # -----------------------------------------
    # 1. Exact Match (EM)
    # -----------------------------------------
    em = np.mean([
        t == p
        for t, p in zip(true_texts, pred_texts)
    ])

    # -----------------------------------------
    # 2. String Distance (SD)
    # normalized Levenshtein similarity
    # -----------------------------------------
    sd = np.mean([
        Levenshtein.normalized_similarity(t, p)
        for t, p in zip(true_texts, pred_texts)
    ])

    # -----------------------------------------
    # 3. Embedding Distance (ED)
    # cosine similarity using sentence embeddings
    # -----------------------------------------
    true_emb = model.encode(
        true_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    pred_emb = model.encode(
        pred_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    ed = np.mean([
        cosine_similarity([a], [b])[0][0]
        for a, b in zip(true_emb, pred_emb)
    ])

    return em, sd, ed

# =========================================================
# STEP 5: Evaluate all folders
# =========================================================

results = []

for folder in os.listdir(base_dir):

    folder_path = os.path.join(base_dir, folder)

    if not os.path.isdir(folder_path):
        continue

    submission_path = os.path.join(folder_path, "submission.csv")

    if not os.path.exists(submission_path):
        print(f"Skipping {folder} (no submission.csv)")
        continue

    print(f"\nEvaluating: {folder}")

    try:

        # -------------------------------------
        # Load submission
        # -------------------------------------
        sub = pd.read_csv(submission_path)

        # Lowercase columns
        sub.columns = sub.columns.str.lower()

        # -------------------------------------
        # Merge by ID
        # -------------------------------------
        merged = target_df.merge(
            sub,
            on="id",
            suffixes=("_true", "_pred")
        )

        print(f"Merged rows: {len(merged)}")

        # -------------------------------------
        # FIELD AREA METRICS
        # -------------------------------------
        field_em, field_sd, field_ed = compute_metrics(
            merged["field_area_true"],
            merged["field_area_pred"]
        )

        # -------------------------------------
        # DISCIPLINE AREA METRICS
        # -------------------------------------
        disc_em, disc_sd, disc_ed = compute_metrics(
            merged["discipline_area_true"],
            merged["discipline_area_pred"]
        )

        # -------------------------------------
        # Exact BOTH fields
        # -------------------------------------
        exact_both = np.mean(
            (
                merged["field_area_true"].map(normalize_text)
                ==
                merged["field_area_pred"].map(normalize_text)
            )
            &
            (
                merged["discipline_area_true"].map(normalize_text)
                ==
                merged["discipline_area_pred"].map(normalize_text)
            )
        )



        # -------------------------------------
        # Store results
        # -------------------------------------
        results.append({

            "folder": folder,
            "rows": len(merged),

            # Field metrics
            "field_em": round(field_em, 4),
            "field_sd": round(field_sd, 4),
            "field_ed": round(field_ed, 4),

            # Discipline metrics
            "discipline_em": round(disc_em, 4),
            "discipline_sd": round(disc_sd, 4),
            "discipline_ed": round(disc_ed, 4),


            # Strict exact row match
            "exact_both_fields": round(exact_both, 4),
        })

    except Exception as e:
        print(f"Error in {folder}: {e}")

# =========================================================
# STEP 6: Final leaderboard
# =========================================================

summary = pd.DataFrame(results)

if len(summary) > 0:

    summary = summary.sort_values(
        "exact_both_fields",
        ascending=False
    )

    print("\n================ LEADERBOARD ================\n")
    print(summary)

    # Save CSV
    output_path = os.path.join(base_dir, "metrics_summary.csv")

    summary.to_csv(output_path, index=False)

    print(f"\nSaved results to:\n{output_path}")

else:
    print("\nNo valid submissions found.")

Renamed: submission - Ella Hawkins -> Ella Hawkins
Renamed: submission - David Condrey -> David Condrey
Renamed: submission - Edwin Thuma -> Edwin Thuma

Folder renaming complete.

Target columns:
['id', 'field_area', 'discipline_area']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Evaluating: Ella Hawkins
Merged rows: 3106

Evaluating: David Condrey
Merged rows: 3106

Evaluating: Edwin Thuma
Merged rows: 3106

Evaluating: gemini-3.5-flash - Jaap Kamps
Merged rows: 3106

Evaluating: st26-Task3-dummy - Jaap Kamps
Merged rows: 3106

================ LEADERBOARD ================

                          folder  rows  field_em  field_sd  field_ed  \
2                    Edwin Thuma  3106    0.9549    0.9798    0.9828   
0                   Ella Hawkins  3106    0.9559    0.9801    0.9824   
3  gemini-3.5-flash - Jaap Kamps  3106    0.8606    0.9373    0.9379   
1                  David Condrey  3106    0.8117    0.9171    0.9180   
4  st26-Task3-dummy - Jaap Kamps  3106    0.7675    0.8993    0.9070   

   discipline_em  discipline_sd  discipline_ed  avg_em  avg_sd  avg_ed  \
2         0.8947         0.9171         0.9411  0.9248  0.9485  0.9619   
0         0.8902         0.9147         0.9386  0.9231  0.9474  0.9605   
3         0.7888         0.8350         0.8